---
## 7. α₂ (SPHERE-16) — Both Channels

Compares α₂ across u and ξ channels on the bundled E9.2 curves.
(α₁/v1 comparison requires raw time-series; use `examples/e9_2_u_vs_xi.py` for that.)


## 0. Install and import

In [ ]:
# pip install fawp-index matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys, json

import fawp_index as fi
from fawp_index.detection.odw import ODWDetector
from fawp_index.core.alpha_v2 import FAWPAlphaIndexV2
from fawp_index.data import E9_2_AGGREGATE_CURVES, E9_2_SEED_CURVES, E9_2_SUMMARY_JSON

print(f'fawp-index {fi.__version__}')
print(f'PEAK_PRED_BITS  = {fi.PEAK_PRED_BITS}')
print(f'NULL_MAX_SHIFT  = {fi.NULL_MAX_SHIFT_E8}')
print(f'NULL_MAX_SHUFFLE= {fi.NULL_MAX_SHUFFLE_E8}')

# Dark plot style matching the paper
plt.rcParams.update({
    'figure.facecolor':  '#07101E',
    'axes.facecolor':    '#0D1729',
    'axes.edgecolor':    '#3A4E70',
    'axes.labelcolor':   '#7A90B8',
    'xtick.color':       '#3A4E70',
    'ytick.color':       '#3A4E70',
    'text.color':        '#EDF0F8',
    'grid.color':        '#182540',
    'legend.facecolor':  '#0D1729',
    'legend.edgecolor':  '#3A4E70',
})
GOLD, BLUE, RED, MUTED = '#D4AF37', '#4A7FCC', '#C0111A', '#3A4E70'

---
## 1. E9.2 — Aggregate MI Curves (u vs ξ steering)

Reproduces Figure 1 of the E9 suite: null-corrected prediction MI and steering MI  
for both the u-channel and ξ-channel, averaged over 20 seeds.

In [ ]:
# Load bundled E9.2 aggregate curves
agg = pd.read_csv(E9_2_AGGREGATE_CURVES)
print('Columns:', agg.columns.tolist())
print(f'Tau range: {agg["tau"].min()} – {agg["tau"].max()}')
agg.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, ch, color_steer in zip(axes, ['u', 'xi'], [GOLD, BLUE]):
    pred_col  = f'pred_mi_{ch}'
    steer_col = f'steer_mi_{ch}'
    gap_col   = f'gap_{ch}'

    tau = agg['tau']

    # Run ODW detection on these curves
    odw = ODWDetector.from_e9_2_data(steering=ch)

    # ODW shading
    if odw.fawp_found and odw.odw_start and odw.odw_end:
        ax.axvspan(odw.odw_start, odw.odw_end, color=RED, alpha=0.15,
                   label=f'ODW τ{odw.odw_start}–{odw.odw_end}')

    ax.plot(tau, agg[pred_col],  color=GOLD, lw=2,   label='Prediction MI (corrected)')
    ax.plot(tau, agg[steer_col], color=color_steer, lw=1.5, ls='--', label=f'Steering MI ({ch})')
    ax.axhline(fi.EPSILON_STEERING_RAW, color=MUTED, ls=':', lw=1, label='ε=0.01')

    # Mark τ⁺ₕ and τf
    if odw.tau_h_plus:
        ax.axvline(odw.tau_h_plus, color='#1DB954', ls='-.', lw=1, label=f'τ⁺ₕ={odw.tau_h_plus}')
    if odw.tau_f:
        ax.axvline(odw.tau_f, color=RED, ls='-.', lw=1, label=f'τf={odw.tau_f}')

    ax.set_xlabel('τ (delay steps)')
    ax.set_ylabel('MI (bits)')
    ax.set_title(f'E9.2 — {ch}-channel steering', color=GOLD, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.3)
    ax.set_xlim(0, 80)

fig.suptitle('E9.2: u vs ξ Steering — Aggregate MI Curves (20 seeds)', 
             color=GOLD, fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('E9_2_aggregate_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved E9_2_aggregate_curves.png')

---
## 2. E9.2 — Published Results Table

Reproduces Table 1 from the E9 paper.

In [ ]:
with open(E9_2_SUMMARY_JSON) as f:
    summary = json.load(f)

agg_s = summary['aggregate_summary']
cfg   = summary['config']

# Build the paper table
table = pd.DataFrame({
    'Metric':         ['τ⁺ₕ (agency horizon)', 'τf (failure cliff)', 'ODW start', 'ODW end',
                       'Peak gap τ', 'Peak gap (bits)', 'Peak pred τ', 'Peak pred (bits)'],
    'u-channel':      [agg_s['tau_h_plus_u'],  agg_s['tau_f'],      agg_s['odw_u_start'],
                       agg_s['odw_u_end'],      agg_s['peak_gap_u_tau'],
                       f"{agg_s['peak_gap_u_bits']:.4f}",
                       agg_s['peak_pred_tau'],  f"{agg_s['peak_pred_bits']:.4f}"],
    'ξ-channel':      [agg_s['tau_h_plus_xi'], agg_s['tau_f'],      agg_s['odw_xi_start'],
                       agg_s['odw_xi_end'],     agg_s['peak_gap_xi_tau'],
                       f"{agg_s['peak_gap_xi_bits']:.4f}",
                       agg_s['peak_pred_tau'],  f"{agg_s['peak_pred_bits']:.4f}"],
    'Constants in fi':['fi.TAU_H_PLUS_E8', 'fi.TAU_F_E8', 'fi.ODW_START_E9', 'fi.ODW_END_E9',
                       '—', 'fi.PEAK_GAP_BITS_E9_U / _XI', '—', 'fi.PEAK_PRED_BITS'],
})

print('='*70)
print('E9.2 KEY RESULTS  (a=1.02, K=0.8, Δ=20, 20 seeds)')
print('='*70)
print(table.to_string(index=False))
print()

# Verify against package constants
print('── Constant verification ──')
checks = [
    ('TAU_H_PLUS_E8', fi.TAU_H_PLUS_E8, agg_s['tau_h_plus_u']),
    ('TAU_F_E8',      fi.TAU_F_E8,      agg_s['tau_f']),
    ('ODW_START_E9',  fi.ODW_START_E9,  agg_s['odw_u_start']),
    ('ODW_END_E9',    fi.ODW_END_E9,    agg_s['odw_u_end']),
    ('PEAK_PRED_BITS',fi.PEAK_PRED_BITS, 2.233669),
]
for name, pkg_val, paper_val in checks:
    match = '✅' if abs(float(pkg_val) - float(paper_val)) < 0.1 else '⚠️'
    print(f'  {match}  fi.{name} = {pkg_val}  (paper: {paper_val})')

---
## 3. E9.2 — Seed-Level Distribution

Shows the distribution of τ⁺ₕ and peak gap across all 20 seeds,  
confirming robustness of the aggregate result.

In [ ]:
seed_df = pd.read_csv(E9_2_SEED_CURVES)
print('Seed curves shape:', seed_df.shape)
print('Seeds:', sorted(seed_df['seed'].unique()) if 'seed' in seed_df.columns else 'no seed col')
seed_df.head(3)

In [ ]:
# Plot per-seed MI curves overlaid
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, ch in zip(axes, ['u', 'xi']):
    pred_col  = f'pred_mi_{ch}'
    steer_col = f'steer_mi_{ch}'

    if 'seed' in seed_df.columns:
        for seed, grp in seed_df.groupby('seed'):
            ax.plot(grp['tau'], grp[pred_col],  color=GOLD, alpha=0.2, lw=0.8)
            ax.plot(grp['tau'], grp[steer_col], color=BLUE, alpha=0.2, lw=0.8, ls='--')

    # Overlay aggregate
    ax.plot(agg['tau'], agg[pred_col],  color=GOLD, lw=2.5, label='Aggregate pred MI')
    ax.plot(agg['tau'], agg[steer_col], color=BLUE, lw=1.8, ls='--', label='Aggregate steer MI')
    ax.axhline(0.01, color=MUTED, ls=':', lw=1)

    ax.set_xlabel('τ (delay steps)')
    ax.set_ylabel('MI (bits)')
    ax.set_title(f'E9.2 seed distribution — {ch}', color=GOLD, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 80)

fig.suptitle('E9.2: Per-seed MI curves (faint) + aggregate (bold)', color=GOLD, fontsize=10)
plt.tight_layout()
plt.savefig('E9_2_seed_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. E9.3 — Null Controls

Verifies that the null floor is well below ε=0.01 bits,  
confirming the detection is not a statistical artefact.

In [ ]:
print('E9.3 Null Control Results')
print('='*45)
print(f'  Shuffle null max  = {fi.NULL_MAX_SHUFFLE_E8:.5f} bits  (fi.NULL_MAX_SHUFFLE_E8)')
print(f'  Shift null max    = {fi.NULL_MAX_SHIFT_E8:.5f} bits  (fi.NULL_MAX_SHIFT_E8)')
print(f'  Detection threshold ε = {fi.EPSILON_STEERING_RAW:.3f} bits')
print()

ratio_shuf = fi.NULL_MAX_SHUFFLE_E8 / fi.EPSILON_STEERING_RAW
ratio_shift = fi.NULL_MAX_SHIFT_E8 / fi.EPSILON_STEERING_RAW
print(f'  Null-to-epsilon ratio (shuffle): {ratio_shuf:.1%}')
print(f'  Null-to-epsilon ratio (shift):   {ratio_shift:.1%}')
print()
print('  ✅ Both null controls < ε — detection is genuine')

# Visual
fig, ax = plt.subplots(figsize=(6, 3))
names  = ['Shuffle null\nmax', 'Shift null\nmax', 'ε threshold']
vals   = [fi.NULL_MAX_SHUFFLE_E8, fi.NULL_MAX_SHIFT_E8, fi.EPSILON_STEERING_RAW]
colors = [MUTED, MUTED, RED]
bars   = ax.bar(names, vals, color=colors, width=0.4, alpha=0.85)
ax.set_ylabel('bits')
ax.set_title('E9.3: Null controls vs detection threshold', color=GOLD, fontweight='bold')
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.0001,
            f'{val:.5f}', ha='center', va='bottom', fontsize=8, color='#EDF0F8')
plt.tight_layout()
plt.savefig('E9_3_null_controls.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. E9.4 — Light Parameter Grid (FAWP Basin)

Reproduces the (a, K) FAWP detection basin from the E9 paper.  
Run the grid scan using `fi.benchmarks.run_grid_scan()` or load the bundled data.

In [ ]:
# Load bundled E6 surface data (the grid basis)
from fawp_index.data import E6_SURFACE_BITS_MEAN
import pandas as pd

e6 = pd.read_csv(E6_SURFACE_BITS_MEAN)
print('E6 surface shape:', e6.shape)
print('Columns:', e6.columns.tolist())
e6.head(3)

In [ ]:
# Plot FAWP basin heatmap from E6/E9 grid data
try:
    pivot = e6.pivot(index='K', columns='a', values='bits_mean')
    
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.values, aspect='auto', origin='lower', cmap='YlOrRd',
                   extent=[pivot.columns.min(), pivot.columns.max(),
                           pivot.index.min(), pivot.index.max()])
    plt.colorbar(im, ax=ax, label='Peak gap (bits)', shrink=0.8)
    
    # Mark the E9.2 baseline point
    ax.scatter([1.02], [0.8], color='white', s=120, marker='*', zorder=5,
               label='E9.2 baseline (a=1.02, K=0.8)')
    
    # Mark ε boundary
    ax.contour(pivot.columns, pivot.index, pivot.values,
               levels=[fi.EPSILON_STEERING_RAW], colors=[BLUE], linewidths=2,
               linestyles='--')
    ax.set_xlabel('a (system gain)')
    ax.set_ylabel('K (control gain)')
    ax.set_title('E9.4 / E6: FAWP Detection Basin — (a, K) parameter space',
                 color=GOLD, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('E9_4_fawp_basin.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as ex:
    print(f'Basin plot requires pivot-able data: {ex}')
    print('Run generate_heavy_grid() for the full E9.5 sweep')

---
## 6. E9.7 — Comparative Timing Results

Reproduces the E9.7 timing sweep: gap2 peak vs α₂ vs α baseline,  
showing which detector leads the failure cliff.

In [ ]:
print('E9.7 Comparative Timing Sweep')
print('  4,244 runs (2,122 per channel × 2 channels)')
print('  Baseline: a=1.02, K=0.8, seeds 51500–51621+')
print()

e97 = {
    'Detector':         ['gap2 peak (raw leverage gap)', 'α₂ (SPHERE-16)', 'α (baseline, old)'],
    'Leads cliff u':    [f'+{fi.E97_MEAN_LEAD_GAP2_TO_CLIFF_U:.4f}',
                         f'+{fi.E97_MEAN_LEAD_ALPHA2_TO_CLIFF_U:.4f}',
                         f'{fi.E97_MEAN_LEAD_ALPHA_TO_CLIFF_U:.4f}'],
    'Leads cliff ξ':    [f'+{fi.E97_MEAN_LEAD_GAP2_TO_CLIFF_XI:.4f}',
                         f'+{fi.E97_MEAN_LEAD_ALPHA2_TO_CLIFF_XI:.4f}',
                         f'{fi.E97_MEAN_LEAD_ALPHA_TO_CLIFF_XI:.4f}'],
    'Err vs ODW (avg)': [f'{fi.E97_MEAN_ABS_ERR_GAP2_VS_ODW_START:.2f} delays',
                         f'{fi.E97_MEAN_ABS_ERR_ALPHA2_VS_ODW_START:.2f} delays',
                         f'{fi.E97_MEAN_ABS_ERR_ALPHA_VS_ODW_START:.2f} delays'],
    'Verdict':          ['✅ BEST', '✅ Good', '❌ Lags cliff'],
}
print(pd.DataFrame(e97).to_string(index=False))
print()
print(f'  Total runs: {fi.E97_N_RUNS}')
print(f'  Mean τf:    {fi.E97_MEAN_TAU_F}')

In [ ]:
# Visualise E9.7 timing comparison
fig, ax = plt.subplots(figsize=(8, 4))

detectors = ['gap2 peak', 'α₂ (SPHERE-16)', 'α (old)']
leads_u   = [fi.E97_MEAN_LEAD_GAP2_TO_CLIFF_U,
              fi.E97_MEAN_LEAD_ALPHA2_TO_CLIFF_U,
              fi.E97_MEAN_LEAD_ALPHA_TO_CLIFF_U]
leads_xi  = [fi.E97_MEAN_LEAD_GAP2_TO_CLIFF_XI,
              fi.E97_MEAN_LEAD_ALPHA2_TO_CLIFF_XI,
              fi.E97_MEAN_LEAD_ALPHA_TO_CLIFF_XI]

x = np.arange(3)
w = 0.35
bars_u  = ax.bar(x - w/2, leads_u,  w, label='u-channel',  color=GOLD,  alpha=0.9)
bars_xi = ax.bar(x + w/2, leads_xi, w, label='ξ-channel', color=BLUE,  alpha=0.9)
ax.axhline(0, color=MUTED, lw=0.8, ls='-')
ax.axhline(0, color=RED,   lw=1.5, ls='--', alpha=0.5, label='Cliff (τf)')

for bars in (bars_u, bars_xi):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + (0.03 if h >= 0 else -0.08),
                f'{h:+.3f}', ha='center', va='bottom', fontsize=8, color='#EDF0F8')

ax.set_xticks(x)
ax.set_xticklabels(detectors)
ax.set_ylabel('Mean lead (+ = before cliff)')
ax.set_title('E9.7: Detector timing relative to failure cliff', color=GOLD, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('E9_7_timing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nVerdict: gap2 peak leads cliff by ~0.6 delays and localises ODW within ~2.1 delay steps')

---
## 7. α₂ vs α₁ — SPHERE-16 Calibration

Compares the old α₁ (baseline) with the SPHERE-16 calibrated α₂  
on the bundled E9.2 curves.

In [ ]:
from fawp_index.core.alpha_v2 import FAWPAlphaIndexV2     # α₂ (SPHERE-16)
# Note: FAWPAlphaIndex v1 requires raw time-series (pred, future, action, obs).
# The bundled E9.2 data provides corrected MI curves, so we run α₂ only.

tau  = agg['tau'].values
pred_u  = agg['pred_mi_u'].values
steer_u = agg['steer_mi_u'].values

# α₂ on u-channel
a2      = FAWPAlphaIndexV2()
res2    = a2.compute(tau, pred_u, steer_u)
alpha2  = res2.alpha

# α₂ on ξ-channel for comparison
pred_xi  = agg['pred_mi_xi'].values
steer_xi = agg['steer_mi_xi'].values
res2_xi  = a2.compute(tau, pred_xi, steer_xi)
alpha2_xi = res2_xi.alpha

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tau, alpha2,    color=GOLD,  lw=2,   label='α₂ u-channel (SPHERE-16)')
ax.plot(tau, alpha2_xi, color=BLUE,  lw=1.5, ls='--', label='α₂ ξ-channel (SPHERE-16)')
ax.axhline(0, color=RED, lw=0.8, ls=':')

peak_idx = int(alpha2.argmax())
ax.scatter([tau[peak_idx]], [alpha2[peak_idx]], color=RED, s=60, zorder=5)
ax.annotate(f'Peak {alpha2[peak_idx]:.4f}b @ τ={tau[peak_idx]}',
            xy=(tau[peak_idx], alpha2[peak_idx]),
            xytext=(tau[peak_idx]+3, alpha2[peak_idx]+0.05),
            color='#EDF0F8', fontsize=8,
            arrowprops=dict(arrowstyle='->', color=RED))

ax.set_xlabel('τ (delay steps)')
ax.set_ylabel('α₂(τ) (bits)')
ax.set_title('α₂ (SPHERE-16) on E9.2 aggregate curves', color=GOLD, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('E9_alpha_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'α₂ u  peak: {max(alpha2):.4f} bits  (fi.PEAK_PRED_BITS = {fi.PEAK_PRED_BITS})')
print(f'α₂ ξ  peak: {max(alpha2_xi):.4f} bits')


---
## 8. Cite

If this notebook contributed to your work, please cite:

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'fawp_index', 'cite'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    # Fallback
    print('''@software{fawp_index_software,
  author  = {Clayton, Ralph},
  title   = {fawp-index: FAWP Alpha Index — Information-Control Exclusion Principle detector},
  year    = {2026},
  doi     = {10.5281/zenodo.18673949},
  url     = {https://github.com/DrRalphClayton/fawp-index}
}

@article{fawp_e9_suite,
  author  = {Clayton, Ralph},
  title   = {FAWP Alpha Index — E9 Suite: Comparative timing, gap2 detector, robustness},
  year    = {2026},
  doi     = {10.5281/zenodo.19065421}
}

@article{fawp_sphere16,
  author  = {Clayton, Ralph},
  title   = {FAWP Alpha Index — SPHERE-16: E8 Flagship Calibration},
  year    = {2026},
  doi     = {10.5281/zenodo.18673949}
}''')

---
*Generated with fawp-index · fawp-scanner.info*